# V011 final results — official MAE 0.18025

The new five-member ensemble improved the official score from **0.18550 to 0.18025 (2.83%)**. Both submissions, all checkpoints and the receipts remain preserved.

The new models contribute different views of a breath: curve consistency, learned response memory, and a shared pressure offset. Their fixed average with the original compact/deep models reduced error on both grouped validations. This does not establish that any single addition caused the full improvement.

The A median absolute error of 0.09816 describes typical individual errors. It is not a mean MAE or official score. Mean MAE 0.10 has not been achieved.


In [1]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd()
receipt = json.loads((root / "research/submission_receipt_v011_final.json").read_text())
D = json.loads((root / "outputs/confirmation_v011/confirmation_result.json").read_text())
rows = [
    ("Grouped A", .19656260982217663, receipt["A_validation_MAE"]),
    ("Fresh grouped D", D["decision"]["baseline_MAE"], receipt["D_confirmation_MAE"]),
    ("Official Gym", receipt["prior_best_official_MAE"], receipt["official_MAE"]),
]
results = pd.DataFrame(rows, columns=["Evaluation", "Previous mean MAE", "New mean MAE"])
results["Reduction %"] = 100 * (1 - results["New mean MAE"] / results["Previous mean MAE"])
print(results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("Compare old and new within each row; each row uses different breaths.")


     Evaluation  Previous mean MAE  New mean MAE  Reduction %
      Grouped A           0.196563      0.190851     2.905731
Fresh grouped D           0.203509      0.196962     3.216717
   Official Gym           0.185500      0.180250     2.830189
Compare old and new within each row; each row uses different breaths.


In [2]:
manifest = json.loads((root / "outputs/confirmation_v011/final_all/final_manifest.json").read_text())
audit = json.loads((root / "research/final_review_v011.json").read_text())
rows = []
for name, member in manifest["members"].items():
    rows.append((name, manifest["weights"][name], member["recipe"]["selected_epochs"],
                 member["training_breaths"], member["elapsed_seconds"] / 60,
                 member["reused_accepted_all_training_checkpoint"]))
print(pd.DataFrame(rows, columns=["Member", "Weight", "Epochs", "Breaths", "New fit minutes", "Reused accepted checkpoint"]).to_string(index=False))
print(f"Final artifact audit: {audit['check_count']}/{audit['check_count']} passed.")
print("Five additional full optimizer/loss/config comparisons passed before submission.")
print("Submission rows:", receipt["submission_rows"])
print("Submission SHA256:", receipt["submission_sha256"])
print("Final refit has an independent holdout:", receipt["final_refit_has_independent_holdout"])
print("GPU released:", receipt["GPU_released"])


          Member  Weight  Epochs  Breaths  New fit minutes  Reused accepted checkpoint
            deep   0.500     189    67905         0.000000                        True
original_compact   0.125     193    67905         0.000000                        True
           curve   0.125     193    67905        38.854800                       False
          memory   0.125     191    67905        47.923221                       False
          offset   0.125     193    67905        41.900094                       False
Final artifact audit: 146/146 passed.
Five additional full optimizer/loss/config comparisons passed before submission.
Submission rows: 603600
Submission SHA256: c344a485f5ef1290aadf5dfd379333009efab674b7ccdff6f361e225a4b8b77a
Final refit has an independent holdout: False
GPU released: True


## Evidence and limits

- Fresh D confirmation cold-fitted all five members. Its whole-breath 95% interval for candidate minus reference MAE was **[-0.00801008, -0.00489360]**. D is resampling evidence after repeated research on A.
- D audit: `research/confirmation_review_v011.json` (211 checks). Final audit: `research/final_review_v011.json` (146 checks), plus five full configuration comparisons before submission.
- Training used only the approved Gym data. All-data final refits used 67,905 breaths and frozen A-selected epoch/LR budgets. No D-based weight or epoch search.
- A genuine legacy lock digest convention error stopped the first final attempt before fitting. Its verifier was corrected; old sources/logs remain preserved. No model recipes or D outcomes changed.
- Official receipt: `research/submission_receipt_v011_final.json`. CLI log: `outputs/confirmation_v011/final_all/submission_cli.log`.
- Verified CSV: `outputs/confirmation_v011/final_all/submission.csv`. Exact 603,600 test/sample IDs, finite native predictions, and fixed weights were checked.
- Both original and recovery archives are retained. No running or queued GPU work remains as of the latest DataHub check.
